In [4]:
"""
Fair-Projected ParPIC + Fair Assignment (rounding) step
=========================================================
Why Fair-Projected ParPIC alone can tie or lose to unconstrained ParPIC
------------------------------------------------------------------------
Proposition 1 guarantees F^T Z^(t) = 0 exactly. Unpacking what that says
for a single sensitive attribute with groups S_1..S_G: for every embedding
column z and every group j,
        sum_i (1{i in S_j} - alpha_j) z_i = 0
   <=>  mean_{i in S_j} z_i  =  mean_{all i} z_i ,
i.e. the Pi_F projection makes every group's *centroid* in embedding space
coincide with the global centroid (verified numerically in this script:
||mean_F - mean_M|| ~ 1e-16 after projection). It is a first-moment
(centroid-parity) condition on the embedding.

Lemma 1 says that a *cluster-indicator* vector y satisfying F^T y = 0 is
perfectly balanced -- but the final clustering is produced by k-means, and
the k-means indicator vectors are NOT in the column span of Z, so they are
not covered by the guarantee. Two groups can have identical centroids
while one small region of the embedding is still dominated by one group;
k-means is free to carve that region into its own cluster. This is the
same "locally segregated pocket" failure mode documented in the paper's
Limitations section, and it is where the balance is lost.

The fix: make the discretization step fair too
------------------------------------------------
The loss happens in the rounding step, so we constrain the rounding step:
replace plain k-means assignment with a *fairness-constrained assignment*
(Bera, Chakrabarty, Flores & Negahbani, "Fair Algorithms for Clustering",
NeurIPS 2019). Given centers mu_1..mu_k in the embedding, solve

   min_x  sum_{i,c} x_ic ||z_i - mu_c||^2
   s.t.   sum_c x_ic = 1                                  (each point once)
          (1-delta) alpha_g  <=  |C_c cap S_g| / |C_c|  <=  alpha_g/(1-delta)
                                                          (every cluster c,
                                                           every group g)
          x_ic in {0,1}

as a mixed-integer program (HiGHS), then update the centers to the means
of their assigned points and repeat (Lloyd-style) until assignments stop
changing. By construction every cluster's group fraction is at least
(1-delta) alpha_g, so

        Balance  >=  (1 - delta) * min_g alpha_g        (a hard guarantee)

delta is the user-set tolerance: delta -> 0 forces exact proportional
representation in every cluster; larger delta trades balance for
compactness.

What Pi_F still contributes
----------------------------
The fair assignment step can be run on either embedding. The balance
guarantee comes from the assignment constraints, so the question for
Pi_F becomes: does a centroid-aligned embedding make fairness *cheaper*?
We measure the "price of fairness" = relative increase in within-cluster
sum of squares (WCSS) caused by enforcing the constraints, on each
embedding, and report it alongside the balance.

Everything upstream (data cleaning, K-NN digraph, P-RW operator, entropy-
based diffusion time, Definition 1 / Pi_F, elbow selection of k and
n_neighbors, balance_objective) is unchanged from the earlier scripts.
Fully self-contained (no cross-file imports).
"""

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.optimize import milp, LinearConstraint, Bounds
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import KMeans


# ======================================================================
# 1. Data cleaning (same recipe as every earlier script, generalized to an
#    arbitrary sensitive attribute / set of valid categories)
# ======================================================================
def clean_and_filter_data(df, sensitive_col, valid_categories, missing_threshold=0.4):
    df = df.replace("?", np.nan)
    df = df.replace("unknown", np.nan)
    df = df[df[sensitive_col].isin(valid_categories)]

    missing_fraction = df.isnull().mean()
    cols_to_drop = missing_fraction[missing_fraction > missing_threshold].index
    df = df.drop(columns=cols_to_drop)

    numeric_cols = df.select_dtypes(exclude=["object"]).columns
    df = df[list(numeric_cols) + [sensitive_col]]

    for col in numeric_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
    return df


def prepare_numeric_features(df, sensitive_col):
    sensitive = df[sensitive_col].astype(str)
    groups, group_ids = np.unique(sensitive, return_inverse=True)
    X = df.drop(columns=[sensitive_col]).values
    X = StandardScaler().fit_transform(X)
    return X, group_ids, groups


def load_dataset(csv_path, sensitive_col, valid_categories, sep=",",
                 n_sample=1500, seed=42):
    df = pd.read_csv(csv_path, sep=sep)
    df = clean_and_filter_data(df, sensitive_col, valid_categories)
    if n_sample is not None and n_sample < len(df):
        parts = []
        for _, g in df.groupby(sensitive_col):
            n_g = min(len(g), int(round(n_sample * len(g) / len(df))))
            parts.append(g.sample(n=n_g, random_state=seed))
        df = pd.concat(parts, axis=0).reset_index(drop=True)
    return prepare_numeric_features(df, sensitive_col)


# ======================================================================
# 2. Generic knee finder (max distance to chord), used for diffusion time
#    and for both elbow selections
# ======================================================================
def find_knee(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if y.max() - y.min() < 1e-12:
        return int(x[0])
    xn = (x - x.min()) / (x.max() - x.min())
    yn = (y - y.min()) / (y.max() - y.min())
    x0, y0, x1, y1 = xn[0], yn[0], xn[-1], yn[-1]
    num = np.abs((y1 - y0) * xn - (x1 - x0) * yn + x1 * y0 - y1 * x0)
    den = np.sqrt((y1 - y0) ** 2 + (x1 - x0) ** 2)
    return int(x[int(np.argmax(num / (den if den > 1e-12 else 1.0)))])


# ======================================================================
# 3. ParPIC machinery (unchanged)
# ======================================================================
def build_knn_digraph(X, n_neighbors):
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(X)
    W = nn.kneighbors_graph(X, mode="connectivity")
    W.setdiag(0)
    W.eliminate_zeros()
    return sp.csr_matrix(W)


def natural_random_walk(W):
    d_out = np.asarray(W.sum(axis=1)).flatten()
    d_in = np.asarray(W.sum(axis=0)).flatten()
    d_out_safe = np.where(d_out > 0, d_out, 1.0)
    return sp.csr_matrix(sp.diags(1.0 / d_out_safe) @ W), d_out, d_in


def degree_vertex_measure(d_out, d_in, gamma=0.5):
    nu = gamma * d_in / d_in.sum() + (1.0 - gamma) * d_out / d_out.sum()
    return np.maximum(nu, 1e-12)


def build_p_rw_operator(P, nu):
    D_nu = sp.diags(nu)
    numerator = D_nu @ P + P.T @ D_nu
    xi = np.asarray(nu @ P).flatten()
    denom = np.where(nu + xi > 1e-15, nu + xi, 1e-15)
    return sp.csr_matrix(sp.diags(1.0 / denom) @ numerator)


def _row_entropy(row):
    p = row[row > 1e-15]
    return 0.0 if len(p) == 0 else float(-np.sum(p * np.log(p)))


def select_diffusion_time(P_nu, t_max, n_probes, seed=42):
    n = P_nu.shape[0]
    rng = np.random.default_rng(seed)
    probe = rng.choice(n, size=min(n_probes, n), replace=False)
    V = sp.csr_matrix((np.ones(len(probe)), (np.arange(len(probe)), probe)),
                      shape=(len(probe), n))
    H = np.zeros(t_max + 1)
    for t in range(1, t_max + 1):
        V = V @ P_nu
        Vd = V.toarray()
        H[t] = (n / len(probe)) * sum(_row_entropy(Vd[r]) for r in range(Vd.shape[0]))
    return find_knee(np.arange(1, t_max + 1), H[1:])


def build_group_indicator_matrix(group_ids, n_groups):
    n = len(group_ids)
    alpha = np.bincount(group_ids, minlength=n_groups) / n
    F = np.zeros((n, n_groups - 1))
    for j in range(n_groups - 1):
        F[:, j] = (group_ids == j).astype(float) - alpha[j]
    return F, alpha


def build_fairness_projector(F):
    FtF_inv = np.linalg.inv(F.T @ F)
    return lambda M: M - F @ (FtF_inv @ (F.T @ M))


def parpic_embeddings(X, group_ids, n_groups, n_neighbors, gamma=0.5,
                      t_max=40, seed=42):
    """Returns (Z_unconstrained, Z_fair_projected, t_star, max|F^T Z_fair|).
    Both embeddings share the same graph, operator, diffusion time and
    random start Z^(0); the only difference is Pi_F in every step."""
    n = X.shape[0]
    d = int(np.ceil(np.sqrt(n)))
    W = build_knn_digraph(X, n_neighbors)
    P, d_out, d_in = natural_random_walk(W)
    P_nu = build_p_rw_operator(P, degree_vertex_measure(d_out, d_in, gamma))
    t_star = select_diffusion_time(P_nu, t_max, d, seed=seed)

    F, _ = build_group_indicator_matrix(group_ids, n_groups)
    Pi_F = build_fairness_projector(F)

    Z0 = np.random.default_rng(seed).standard_normal((n, d))
    Zb, Zf = Z0.copy(), Z0.copy()
    for _ in range(t_star):
        Zb = P_nu @ Zb
        Zf = Pi_F(P_nu @ Zf)
    return Zb, Zf, t_star, float(np.max(np.abs(F.T @ Zf)))


# ======================================================================
# 4. Elbow selection of n_neighbors and k (unchanged methodology; neither
#    uses the sensitive attribute)
# ======================================================================
def select_n_neighbors_by_elbow(X, candidates=(3, 5, 7, 10, 12, 15, 18, 20, 25, 30)):
    mean_kdist = []
    for k_nn in candidates:
        dist, _ = NearestNeighbors(n_neighbors=k_nn + 1).fit(X).kneighbors(X)
        mean_kdist.append(dist[:, k_nn].mean())
    return find_knee(candidates, mean_kdist)


def select_k_by_elbow(Z, candidates=tuple(range(2, 16)), seed=42):
    inertias = [KMeans(n_clusters=k, n_init=10, random_state=seed).fit(Z).inertia_
                for k in candidates]
    return find_knee(candidates, inertias)


# ======================================================================
# 5. Fairness-constrained assignment (Bera et al., 2019) + Lloyd updates
# ======================================================================
def fair_assignment(Z, centers, group_ids, n_groups, alpha, delta, time_limit=60):
    """Exact MILP: min sum x_ic ||z_i - mu_c||^2 subject to one cluster per
    point and (1-delta)alpha_g <= frac_g(C_c) <= alpha_g/(1-delta), with
    every cluster non-empty. Solved with HiGHS via scipy.optimize.milp."""
    n, k = Z.shape[0], centers.shape[0]
    cost = ((Z[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)  # (n,k)
    scale = cost.mean() if cost.mean() > 0 else 1.0
    c_vec = (cost / scale).ravel()                                    # x_{i,c} at i*k+c

    rows, cols, vals, lb, ub = [], [], [], [], []
    r = 0
    # (a) each point assigned to exactly one cluster
    for i in range(n):
        rows += [r] * k
        cols += list(range(i * k, i * k + k))
        vals += [1.0] * k
        lb.append(1.0); ub.append(1.0); r += 1

    lower = (1.0 - delta) * alpha
    upper = np.minimum(alpha / (1.0 - delta), 1.0)
    idx_all = np.arange(n)
    for c in range(k):
        var = idx_all * k + c
        for g in range(n_groups):
            in_g = (group_ids == g).astype(float)
            # (b) lower bound:  sum_i (1{i in g} - lower_g) x_ic >= 0
            rows += [r] * n; cols += var.tolist(); vals += (in_g - lower[g]).tolist()
            lb.append(0.0); ub.append(np.inf); r += 1
            # (c) upper bound:  sum_i (1{i in g} - upper_g) x_ic <= 0
            rows += [r] * n; cols += var.tolist(); vals += (in_g - upper[g]).tolist()
            lb.append(-np.inf); ub.append(0.0); r += 1
        # (d) non-empty cluster
        rows += [r] * n; cols += var.tolist(); vals += [1.0] * n
        lb.append(1.0); ub.append(np.inf); r += 1

    A = sp.csr_matrix((vals, (rows, cols)), shape=(r, n * k))
    res = milp(c=c_vec, constraints=LinearConstraint(A, lb, ub),
               integrality=np.ones(n * k), bounds=Bounds(0, 1),
               options={"time_limit": time_limit, "disp": False,
                        "mip_rel_gap": 5e-3})   # accept solutions within 0.5% of optimal cost
    # Any returned solution (optimal, or best-found at the gap/time limit) is
    # integer-feasible, so the fairness constraints -- and hence the balance
    # floor -- hold exactly regardless; the gap only affects WCSS optimality.
    if res.x is None:
        raise RuntimeError(f"Fair assignment MILP failed: {res.message}")
    x = res.x.reshape(n, k)
    return np.argmax(x, axis=1)


def fair_kmeans(Z, group_ids, n_groups, k, delta, seed=42, max_iter=10):
    """Lloyd-style fair k-means: init centers with ordinary k-means, then
    alternate (fair assignment MILP) <-> (center = mean of assigned points)
    until the assignment stops changing."""
    alpha = np.bincount(group_ids, minlength=n_groups) / len(group_ids)
    km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(Z)
    centers = km.cluster_centers_
    labels = None
    for _ in range(max_iter):
        new_labels = fair_assignment(Z, centers, group_ids, n_groups, alpha, delta)
        if labels is not None and np.array_equal(new_labels, labels):
            break
        labels = new_labels
        centers = np.vstack([Z[labels == c].mean(axis=0) for c in range(k)])
    return labels


def wcss(Z, labels):
    return float(sum(((Z[labels == c] - Z[labels == c].mean(axis=0)) ** 2).sum()
                     for c in np.unique(labels)))


# ======================================================================
# 6. Balance objective (unchanged)
# ======================================================================
def balance_objective(labels, group_ids):
    vals = []
    for c in np.unique(labels):
        idx_c = labels == c
        for g in np.unique(group_ids):
            vals.append(np.sum(idx_c & (group_ids == g)) / np.sum(idx_c))
    return float(np.min(vals))


# ======================================================================
# 7. One full run on one dataset / seed
# ======================================================================
def run_all_methods(X, group_ids, n_groups, n_neighbors, k, delta, seed=42):
    Zb, Zf, t_star, max_FtZ = parpic_embeddings(X, group_ids, n_groups,
                                                n_neighbors, seed=seed)
    out = {"t_star": t_star, "max_FtZ": max_FtZ}

    for name, Z in [("ParPIC", Zb), ("FP-ParPIC", Zf)]:
        lab_km = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(Z)
        lab_fa = fair_kmeans(Z, group_ids, n_groups, k, delta, seed=seed)
        w_km, w_fa = wcss(Z, lab_km), wcss(Z, lab_fa)
        out[f"{name} + k-means"] = balance_objective(lab_km, group_ids)
        out[f"{name} + fair assign"] = balance_objective(lab_fa, group_ids)
        out[f"{name} price"] = (w_fa - w_km) / w_km   # relative WCSS increase

    m0 = Zf[group_ids == 0].mean(axis=0)
    m_all = Zf.mean(axis=0)
    out["centroid_gap_fair"] = float(np.linalg.norm(m0 - m_all))
    return out


# ======================================================================
# 8. Main: three datasets, elbow-selected (k, n_neighbors), multi-seed
# ======================================================================
DATASETS = [
    ("diabetic_data", "diabetic_data.csv", "gender", ("Male", "Female"), ","),
    ("uci_census",    "uci_census.csv",    "gender", ("Male", "Female"), ","),
    ("bank_full",     "bank-full.csv",     "marital", ("married", "single", "divorced"), ";"),
]


def main(n_sample=10000, delta=0.05, seeds=(42, 0, 1, 2, 3)):
    methods = ["ParPIC + k-means", "FP-ParPIC + k-means",
               "ParPIC + fair assign", "FP-ParPIC + fair assign"]
    summary = {}

    for name, path, sens, cats, sep in DATASETS:
        print(f"\n{'=' * 72}\n{name}  (sensitive = {sens}, delta = {delta})")
        # hyperparameters chosen once, on the seed-42 subsample, by elbow;
        # then held fixed across seeds so seeds only vary the subsample and
        # the random start
        X, gid, groups = load_dataset(path, sens, cats, sep, n_sample, seed=42)
        G = len(groups)
        alpha = np.bincount(gid, minlength=G) / len(gid)
        nn_star = select_n_neighbors_by_elbow(X)
        Zb, _, _, _ = parpic_embeddings(X, gid, G, nn_star, seed=42)
        k_star = select_k_by_elbow(Zb)
        floor = (1 - delta) * alpha.min()
        print(f"groups={list(groups)}, alpha={np.round(alpha, 4).tolist()}, "
              f"elbow n_neighbors={nn_star}, elbow k={k_star}")
        print(f"guaranteed balance floor for fair assignment = "
              f"(1-delta)*min alpha = {floor:.4f}  (ceiling = min alpha = {alpha.min():.4f})")

        rows = []
        for s in seeds:
            X, gid, _ = load_dataset(path, sens, cats, sep, n_sample, seed=s)
            r = run_all_methods(X, gid, G, nn_star, k_star, delta, seed=s)
            rows.append(r)
            print(f"  seed {s:>2}: t*={r['t_star']:>2}  " +
                  "  ".join(f"{m.split(' + ')[0][:9]:>9}/{m.split(' + ')[1][:4]}={r[m]:.4f}"
                            for m in methods) +
                  f"  |F^TZ|={r['max_FtZ']:.1e}"
                  f"  price: ParPIC {100*r['ParPIC price']:.1f}% / "
                  f"FP-ParPIC {100*r['FP-ParPIC price']:.1f}%", flush=True)
            import json
            with open("fairassign_results.json", "a") as fh:
                fh.write(json.dumps({"dataset": name, "seed": s, **r}) + "\n")

        summary[name] = {"k": k_star, "n_neighbors": nn_star, "floor": floor,
                         "ceiling": alpha.min()}
        for m in methods + ["ParPIC price", "FP-ParPIC price"]:
            v = np.array([r[m] for r in rows])
            summary[name][m] = (v.mean(), v.std())

    print(f"\n{'=' * 72}\nSUMMARY  (mean ± std over {len(seeds)} seeds, delta = {delta})")
    for name, s in summary.items():
        print(f"\n{name}: k={s['k']}, n_neighbors={s['n_neighbors']}, "
              f"floor={s['floor']:.4f}, ceiling={s['ceiling']:.4f}")
        for m in methods:
            print(f"   {m:<26} balance = {s[m][0]:.4f} ± {s[m][1]:.4f}")
        print(f"   price of fairness (rel. WCSS increase): "
              f"ParPIC {100 * s['ParPIC price'][0]:.1f}% ± {100 * s['ParPIC price'][1]:.1f}%,  "
              f"FP-ParPIC {100 * s['FP-ParPIC price'][0]:.1f}% ± {100 * s['FP-ParPIC price'][1]:.1f}%")
    return summary


if __name__ == "__main__":
    main()


diabetic_data  (sensitive = gender, delta = 0.05)
groups=['Female', 'Male'], alpha=[0.5376, 0.4624], elbow n_neighbors=12, elbow k=9
guaranteed balance floor for fair assignment = (1-delta)*min alpha = 0.4393  (ceiling = min alpha = 0.4624)
  seed 42: t*= 9     ParPIC/k-me=0.4159  FP-ParPIC/k-me=0.4246     ParPIC/fair=0.4416  FP-ParPIC/fair=0.4403  |F^TZ|=6.6e-13  price: ParPIC 0.8% / FP-ParPIC 0.4%
  seed  0: t*= 9     ParPIC/k-me=0.4231  FP-ParPIC/k-me=0.4231     ParPIC/fair=0.4423  FP-ParPIC/fair=0.4423  |F^TZ|=7.1e-13  price: ParPIC 0.2% / FP-ParPIC 0.2%
  seed  1: t*= 9     ParPIC/k-me=0.4339  FP-ParPIC/k-me=0.4279     ParPIC/fair=0.4484  FP-ParPIC/fair=0.4492  |F^TZ|=6.6e-13  price: ParPIC 0.6% / FP-ParPIC 0.3%
  seed  2: t*= 9     ParPIC/k-me=0.4313  FP-ParPIC/k-me=0.4366     ParPIC/fair=0.4419  FP-ParPIC/fair=0.4394  |F^TZ|=6.9e-13  price: ParPIC 1.4% / FP-ParPIC 1.2%
  seed  3: t*= 9     ParPIC/k-me=0.4128  FP-ParPIC/k-me=0.4148     ParPIC/fair=0.4398  FP-ParPIC/fair=0.4396  